# 02 - DuckDB Warehouse

This notebook creates a local DuckDB analytical warehouse for the Amsterdam Inside Airbnb dataset.

The purpose of this notebook is to load the cleaned listing master dataset into DuckDB and prepare SQL-based analytical tables for reporting, EDA, and future dashboarding.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.4


In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "amsterdam"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse"

WAREHOUSE_PATH.mkdir(parents=True, exist_ok=True)

DUCKDB_DATABASE_PATH = WAREHOUSE_PATH / "airbnb_market.duckdb"
LISTING_MASTER_FINAL_PATH = PROCESSED_DATA_PATH / "listing_master_final.parquet"

print("DuckDB database path:", DUCKDB_DATABASE_PATH)
print("Listing master path exists:", LISTING_MASTER_FINAL_PATH.exists())
print("Listing master path:", LISTING_MASTER_FINAL_PATH)

DuckDB database path: ..\warehouse\airbnb_market.duckdb
Listing master path exists: True
Listing master path: ..\data\processed\amsterdam\listing_master_final.parquet


In [3]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

print("Connected to DuckDB successfully.")

Connected to DuckDB successfully.


In [4]:
conn.execute("""
CREATE OR REPLACE TABLE listing_master_final AS
SELECT *
FROM read_parquet(?)
""", [str(LISTING_MASTER_FINAL_PATH)])

print("listing_master_final table created successfully in DuckDB.")

listing_master_final table created successfully in DuckDB.


In [5]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,listing_master_final


In [6]:
conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT listing_id) AS unique_listing_ids
FROM listing_master_final
""").fetchdf()

,row_count,unique_listing_ids
0,10465,10465


In [7]:
conn.execute("""
SELECT 
    room_type,
    COUNT(*) AS listing_count,
    ROUND(AVG(price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_price,
    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy
FROM listing_master_final
GROUP BY room_type
ORDER BY listing_count DESC
""").fetchdf()

,room_type,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy
0,Entire home/apt,8489,394.62,331.0,0.2133,0.7867
1,Private room,1929,203.67,171.0,0.4612,0.5388
2,Hotel room,26,249.85,227.5,0.7564,0.2436
3,Shared room,21,80.10,52.0,0.7769,0.2231


In [8]:
conn.close()

print("DuckDB connection closed.")

DuckDB connection closed.


## 2. Star Schema Design

This section creates a simple dimensional model in DuckDB using the final listing master dataset.

The model includes:
- dim_listing
- dim_host
- dim_neighbourhood
- fact_listing_market

The purpose is to separate descriptive attributes from measurable market metrics and support SQL-based analytical queries.

In [9]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

columns_df = conn.execute("""
DESCRIBE listing_master_final
""").fetchdf()

columns_df

,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,listing_name,VARCHAR,YES,None,None,None
2,host_id,BIGINT,YES,None,None,None
3,host_profile_id,BIGINT,YES,None,None,None
4,host_name,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
96,neighbourhood_total_reviews,BIGINT,YES,None,None,None
97,neighbourhood_avg_reviews_per_listing,DOUBLE,YES,None,None,None
98,neighbourhood_total_revenue_proxy,DOUBLE,YES,None,None,None
99,neighbourhood_avg_revenue_proxy,DOUBLE,YES,None,None,None


### Dimension Table: dim_listing

The listing dimension stores descriptive listing and property-level attributes.  
Each row represents one Airbnb listing.

In [10]:
conn.execute("""
CREATE OR REPLACE TABLE dim_listing AS
SELECT
    ROW_NUMBER() OVER (ORDER BY listing_id) AS listing_key,
    listing_id,
    listing_name,
    room_type,
    property_type,
    accommodates,
    bathrooms,
    bathrooms_text,
    bedrooms,
    beds,
    amenities_count,
    instant_bookable,
    latitude,
    longitude,
    city,
    country,
    snapshot_date
FROM listing_master_final
""")

conn.execute("""
SELECT COUNT(*) AS row_count, COUNT(DISTINCT listing_id) AS unique_listing_ids
FROM dim_listing
""").fetchdf()

,row_count,unique_listing_ids
0,10465,10465


### Dimension Table: dim_host

The host dimension stores host-level descriptive attributes.  
Each row represents one unique host.

In [20]:
conn.execute("""
CREATE OR REPLACE TABLE dim_host AS
WITH known_hosts AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY host_id) AS host_key,
        host_id,
        ANY_VALUE(host_profile_id) AS host_profile_id,
        ANY_VALUE(host_name) AS host_name,
        ANY_VALUE(host_since) AS host_since,
        ANY_VALUE(host_is_superhost) AS host_is_superhost,
        ANY_VALUE(host_response_time) AS host_response_time,
        ANY_VALUE(host_response_rate) AS host_response_rate,
        ANY_VALUE(host_acceptance_rate) AS host_acceptance_rate,
        ANY_VALUE(host_listings_count) AS host_listings_count,
        ANY_VALUE(host_total_listings_count) AS host_total_listings_count,
        ANY_VALUE(host_identity_verified) AS host_identity_verified,
        ANY_VALUE(calculated_host_listings_count) AS calculated_host_listings_count,
        ANY_VALUE(host_portfolio_segment) AS host_portfolio_segment
    FROM listing_master_final
    WHERE host_id IS NOT NULL
    GROUP BY host_id
),

unknown_host AS (
    SELECT
        -1 AS host_key,
        NULL AS host_id,
        NULL AS host_profile_id,
        'Unknown Host' AS host_name,
        NULL AS host_since,
        NULL AS host_is_superhost,
        NULL AS host_response_time,
        NULL AS host_response_rate,
        NULL AS host_acceptance_rate,
        NULL AS host_listings_count,
        NULL AS host_total_listings_count,
        NULL AS host_identity_verified,
        NULL AS calculated_host_listings_count,
        'Unknown Host' AS host_portfolio_segment
)

SELECT * FROM unknown_host
UNION ALL
SELECT * FROM known_hosts
""")

conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT host_id) AS unique_host_ids,
    SUM(CASE WHEN host_key = -1 THEN 1 ELSE 0 END) AS unknown_host_records
FROM dim_host
""").fetchdf()

,row_count,unique_host_ids,unknown_host_records
0,9105,9104,1.0


### Dimension Table: dim_neighbourhood

The neighbourhood dimension stores geographic and neighbourhood-level market context.  
Each row represents one Amsterdam neighbourhood.

In [21]:
conn.execute("""
CREATE OR REPLACE TABLE dim_neighbourhood AS
SELECT
    ROW_NUMBER() OVER (ORDER BY neighbourhood) AS neighbourhood_key,
    neighbourhood,
    ANY_VALUE(neighbourhood_group) AS neighbourhood_group,
    ANY_VALUE(city) AS city,
    ANY_VALUE(country) AS country,
    ANY_VALUE(neighbourhood_listing_count) AS neighbourhood_listing_count,
    ANY_VALUE(neighbourhood_unique_hosts) AS neighbourhood_unique_hosts,
    ANY_VALUE(neighbourhood_median_price) AS neighbourhood_median_price,
    ANY_VALUE(neighbourhood_avg_price) AS neighbourhood_avg_price,
    ANY_VALUE(neighbourhood_avg_availability_rate) AS neighbourhood_avg_availability_rate,
    ANY_VALUE(neighbourhood_avg_occupancy_proxy) AS neighbourhood_avg_occupancy_proxy,
    ANY_VALUE(neighbourhood_avg_review_score) AS neighbourhood_avg_review_score,
    ANY_VALUE(neighbourhood_dominant_room_type) AS neighbourhood_dominant_room_type
FROM listing_master_final
WHERE neighbourhood IS NOT NULL
GROUP BY neighbourhood
""")

conn.execute("""
SELECT COUNT(*) AS row_count, COUNT(DISTINCT neighbourhood) AS unique_neighbourhoods
FROM dim_neighbourhood
""").fetchdf()

,row_count,unique_neighbourhoods
0,22,22


### Fact Table: fact_listing_market

The fact table stores listing-level measurable market metrics.

It connects to:
- dim_listing through listing_key
- dim_host through host_key
- dim_neighbourhood through neighbourhood_key

In [22]:
conn.execute("""
CREATE OR REPLACE TABLE fact_listing_market AS
SELECT
    dl.listing_key,
    COALESCE(dh.host_key, -1) AS host_key,
    dn.neighbourhood_key,

    lm.listing_id,
    lm.host_id,
    lm.neighbourhood,

    lm.price_numeric,
    lm.minimum_nights,
    lm.number_of_reviews,
    lm.reviews_per_month,
    lm.availability_365,

    lm.calendar_days,
    lm.available_days,
    lm.unavailable_days,
    lm.availability_rate,
    lm.occupancy_proxy,
    lm.weekend_availability_rate,
    lm.weekday_availability_rate,

    lm.estimated_revenue_proxy,
    lm.listing_price_available_flag,

    lm.detailed_review_count,
    lm.unique_reviewer_count,
    lm.detailed_reviews_last_365d,
    lm.avg_reviews_per_year,
    lm.has_reviews,

    lm.review_scores_rating,
    lm.review_scores_cleanliness,
    lm.review_scores_location,
    lm.review_scores_value
FROM listing_master_final lm
LEFT JOIN dim_listing dl
    ON lm.listing_id = dl.listing_id
LEFT JOIN dim_host dh
    ON lm.host_id = dh.host_id
LEFT JOIN dim_neighbourhood dn
    ON lm.neighbourhood = dn.neighbourhood
""")

conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT listing_id) AS unique_listing_ids,
    COUNT(*) - COUNT(listing_key) AS missing_listing_keys,
    SUM(CASE WHEN host_key IS NULL THEN 1 ELSE 0 END) AS missing_host_keys,
    COUNT(*) - COUNT(neighbourhood_key) AS missing_neighbourhood_keys,
    SUM(CASE WHEN host_key = -1 THEN 1 ELSE 0 END) AS unknown_host_fact_rows
FROM fact_listing_market
""").fetchdf()

,row_count,unique_listing_ids,missing_listing_keys,missing_host_keys,missing_neighbourhood_keys,unknown_host_fact_rows
0,10465,10465,0,0.0,0,96.0


In [23]:
conn.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_host
1,dim_listing
2,dim_neighbourhood
3,fact_listing_market
4,listing_master_final


In [24]:
star_schema_validation = conn.execute("""
SELECT 'dim_listing' AS table_name, COUNT(*) AS row_count FROM dim_listing
UNION ALL
SELECT 'dim_host' AS table_name, COUNT(*) AS row_count FROM dim_host
UNION ALL
SELECT 'dim_neighbourhood' AS table_name, COUNT(*) AS row_count FROM dim_neighbourhood
UNION ALL
SELECT 'fact_listing_market' AS table_name, COUNT(*) AS row_count FROM fact_listing_market
""").fetchdf()

star_schema_validation

,table_name,row_count
0,dim_listing,10465
1,dim_host,9105
2,dim_neighbourhood,22
3,fact_listing_market,10465


In [25]:
REPORTS_PATH = PROJECT_ROOT / "reports" / "data_quality"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

star_schema_validation_output_path = REPORTS_PATH / "amsterdam_star_schema_validation.csv"

star_schema_validation.to_csv(star_schema_validation_output_path, index=False)

print(f"Star schema validation saved to: {star_schema_validation_output_path}")

Star schema validation saved to: ..\reports\data_quality\amsterdam_star_schema_validation.csv


In [26]:
conn.execute("""
SELECT
    dn.neighbourhood,
    dl.room_type,
    COUNT(*) AS listing_count,
    ROUND(AVG(f.price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(f.price_numeric), 2) AS median_price,
    ROUND(AVG(f.availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(f.occupancy_proxy), 4) AS avg_occupancy_proxy,
    ROUND(SUM(f.estimated_revenue_proxy), 2) AS total_revenue_proxy
FROM fact_listing_market f
LEFT JOIN dim_listing dl
    ON f.listing_key = dl.listing_key
LEFT JOIN dim_neighbourhood dn
    ON f.neighbourhood_key = dn.neighbourhood_key
GROUP BY
    dn.neighbourhood,
    dl.room_type
ORDER BY
    total_revenue_proxy DESC
LIMIT 10
""").fetchdf()

,neighbourhood,room_type,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy,total_revenue_proxy
0,De Baarsjes - Oud-West,Entire home/apt,1633,401.29,357.0,0.1922,0.8078,83509991.0
1,De Pijp - Rivierenbuurt,Entire home/apt,1057,423.24,361.0,0.2268,0.7732,48674461.0
2,Centrum-West,Entire home/apt,766,518.39,400.0,0.2647,0.7353,47399839.0
3,Centrum-Oost,Entire home/apt,614,445.48,383.0,0.2976,0.7024,34440350.0
4,Zuid,Entire home/apt,599,459.20,360.0,0.2540,0.7460,32455208.0
5,Westerpark,Entire home/apt,652,354.33,303.5,0.1890,0.8110,29228261.0
6,Oud-Oost,Entire home/apt,571,345.81,304.0,0.1925,0.8075,23498888.0
7,Oud-Noord,Entire home/apt,403,392.91,285.0,0.1828,0.8172,17648085.0
8,Centrum-West,Private room,447,256.67,206.0,0.4610,0.5390,17390977.0
9,Bos en Lommer,Entire home/apt,507,300.40,281.0,0.1557,0.8443,16878453.0


### Star Schema Observations

A simple DuckDB star schema was created using the final listing master dataset.

The model includes listing, host, and neighbourhood dimension tables, plus a listing-level market fact table.  
The fact table stores measurable metrics such as price, availability, occupancy proxy, review activity, review scores, and estimated revenue proxy.

This structure supports SQL-based analytical queries and separates descriptive attributes from measurable business metrics.

### Unknown Host Handling

During star schema validation, 96 fact rows did not have a matching host key because their host_id values were missing in the source listing data.

To preserve referential consistency, an Unknown Host record was added to dim_host with host_key = -1.  
Fact rows with missing host information are assigned to this unknown host key.

This approach preserves all listing records while clearly documenting incomplete host metadata.